# Day 1-01｜Colab 與座標點選工具

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：先不談模型。只學會看一張圖、用滑鼠取得座標，並把座標複製回 Python 變數。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
print(IMAGE_PATH)

In [ ]:
def show_click_tool(image_path, canvas_width=1000, title="click tool"):
    image_path = Path(image_path)
    b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    canvas_id = "canvas_" + image_path.stem.replace("-", "_")
    html = f"""
    <div style="font-family: sans-serif; line-height: 1.5;">
      <h3>{title}</h3>
      <p>請依序點選參考點。每點一次，下方 textarea 會更新座標。完成後把座標複製到下一格的 <code>camera_points</code>。</p>
      <button onclick="{canvas_id}_pts=[]; document.getElementById('{canvas_id}_out').value='[]'; {canvas_id}_draw();">清空</button>
      <br><br>
      <canvas id="{canvas_id}" style="border:1px solid #999; max-width: 100%;"></canvas><br>
      <textarea id="{canvas_id}_out" rows="6" style="width: 95%; font-family: monospace;">[]</textarea>
    </div>
    <script>
      var {canvas_id}_pts = [];
      var {canvas_id}_img = new Image();
      var {canvas_id}_canvas = document.getElementById('{canvas_id}');
      var {canvas_id}_ctx = {canvas_id}_canvas.getContext('2d');
      var {canvas_id}_scale = 1.0;
      function {canvas_id}_draw() {{
        {canvas_id}_ctx.clearRect(0, 0, {canvas_id}_canvas.width, {canvas_id}_canvas.height);
        {canvas_id}_ctx.drawImage({canvas_id}_img, 0, 0, {canvas_id}_canvas.width, {canvas_id}_canvas.height);
        {canvas_id}_ctx.fillStyle = 'red';
        {canvas_id}_ctx.strokeStyle = 'red';
        {canvas_id}_ctx.lineWidth = 2;
        {canvas_id}_ctx.font = '18px sans-serif';
        for (let i=0; i<{canvas_id}_pts.length; i++) {{
          let p = {canvas_id}_pts[i];
          let x = p[0] * {canvas_id}_scale;
          let y = p[1] * {canvas_id}_scale;
          {canvas_id}_ctx.beginPath();
          {canvas_id}_ctx.arc(x, y, 6, 0, 2*Math.PI);
          {canvas_id}_ctx.fill();
          {canvas_id}_ctx.fillText(String(i), x+8, y-8);
        }}
      }}
      {canvas_id}_img.onload = function() {{
        {canvas_id}_scale = Math.min({canvas_width} / {canvas_id}_img.width, 1.0);
        {canvas_id}_canvas.width = Math.round({canvas_id}_img.width * {canvas_id}_scale);
        {canvas_id}_canvas.height = Math.round({canvas_id}_img.height * {canvas_id}_scale);
        {canvas_id}_draw();
      }};
      {canvas_id}_canvas.addEventListener('click', function(e) {{
        const rect = {canvas_id}_canvas.getBoundingClientRect();
        const x = Math.round((e.clientX - rect.left) / {canvas_id}_scale);
        const y = Math.round((e.clientY - rect.top) / {canvas_id}_scale);
        {canvas_id}_pts.push([x, y]);
        document.getElementById('{canvas_id}_out').value = JSON.stringify({canvas_id}_pts);
        {canvas_id}_draw();
      }});
      {canvas_id}_img.src = 'data:image/png;base64,{b64}';
    </script>
    """
    display(HTML(html))


show_click_tool(IMAGE_PATH, title="Camera frame 座標點選工具")

把上方 textarea 的內容貼到下面。例如：`[[818, 480], [1216, 453], ...]`

In [ ]:
# TODO: 把你剛剛點到的座標貼到這裡。
camera_points = [[818, 480], [1216, 453], [1410, 657], [672, 623]]

print("你目前填了", len(camera_points), "個點")
for i, p in enumerate(camera_points):
    print(i, p)

In [ ]:
from src.cv_utils import read_image_rgb, draw_points, show_image

image = read_image_rgb(IMAGE_PATH)
vis = draw_points(
    image, camera_points, labels=[str(i) for i in range(len(camera_points))]
)
show_image(vis, "檢查點位順序")